# Solutions — Patchify / Unpatchify + Timestep Embedding
One set of correct implementations.


In [1]:
import math
import torch
import torch.nn as nn

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def pretty_shapes(**tensors):
    for k, v in tensors.items():
        print(f"{k:>14}: {tuple(v.shape)}")

def assert_close(a, b, tol=1e-6):
    max_err = (a-b).abs().max().item()
    print("max_err:", max_err)
    assert max_err < tol, f"Max error {max_err} >= {tol}"


device: cpu


In [2]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    B, C, H, W = x.shape
    P = patch_size
    assert H % P == 0 and W % P == 0, "H and W must be divisible by patch_size"
    h = H // P
    w = W // P

    x = x.reshape(B, C, h, P, w, P)                 # (B, C, h, P, w, P)
    x = x.permute(0, 2, 4, 1, 3, 5).contiguous()    # (B, h, w, C, P, P)
    tokens = x.reshape(B, h * w, C * P * P)         # (B, N, patch_dim)
    return tokens

def unpatchify(tokens: torch.Tensor, patch_size: int, H: int, W: int, C: int) -> torch.Tensor:
    B, N, D = tokens.shape
    P = patch_size
    assert H % P == 0 and W % P == 0, "H and W must be divisible by patch_size"
    h = H // P
    w = W // P
    assert N == h * w, f"N must equal h*w ({h*w}), got {N}"
    assert D == C * P * P, f"Last dim must be C*P*P ({C*P*P}), got {D}"

    x = tokens.reshape(B, h, w, C, P, P)            # (B, h, w, C, P, P)
    x = x.permute(0, 3, 1, 4, 2, 5).contiguous()    # (B, C, h, P, w, P)
    x = x.reshape(B, C, H, W)                       # (B, C, H, W)
    return x

# test
B, C, H, W = 2, 3, 32, 32
P = 8
x = torch.randn(B, C, H, W, device=device)
tokens = patchify(x, P)
x_recon = unpatchify(tokens, P, H, W, C)
pretty_shapes(x=x, tokens=tokens, x_recon=x_recon)
assert_close(x_recon, x)
print("✅ Patchify/unpatchify passed")


             x: (2, 3, 32, 32)
        tokens: (2, 16, 192)
       x_recon: (2, 3, 32, 32)
max_err: 0.0
✅ Patchify/unpatchify passed


In [3]:
def timestep_embedding(t: torch.Tensor, dim: int, max_period: int = 10_000) -> torch.Tensor:
    assert dim % 2 == 0, "dim must be even"
    t = t.float()
    half = dim // 2

    freqs = torch.exp(
        -math.log(max_period) * torch.arange(0, half, device=t.device).float() / half
    )                                               # (half,)
    args = t[:, None] * freqs[None, :]              # (B, half)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
    return emb

class TimeEmbedding(nn.Module):
    def __init__(self, time_dim: int, model_dim: int):
        super().__init__()
        self.time_dim = time_dim
        self.mlp = nn.Sequential(
            nn.Linear(time_dim, 4 * time_dim),
            nn.SiLU(),
            nn.Linear(4 * time_dim, model_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        emb = timestep_embedding(t, self.time_dim)   # (B, time_dim)
        return self.mlp(emb)                         # (B, model_dim)

# demo: patchify -> proj -> add time embedding (broadcast)
B, C, H, W = 2, 3, 32, 32
P = 8
D = 128

x = torch.randn(B, C, H, W, device=device)
t = torch.randint(0, 1000, (B,), device=device)

tokens = patchify(x, P)
patch_dim = tokens.shape[-1]

proj = nn.Linear(patch_dim, D).to(device)
tok_emb = proj(tokens)                              # (B, N, D)

time_emb = TimeEmbedding(time_dim=64, model_dim=D).to(device)
t_emb = time_emb(t)                                 # (B, D)

conditioned = tok_emb + t_emb[:, None, :]           # (B, N, D)

pretty_shapes(tokens=tokens, tok_emb=tok_emb, t_emb=t_emb, conditioned=conditioned)
print("t_emb[:, None, :]:", t_emb[:, None, :].shape)
print("✅ Time embedding + broadcast passed")


        tokens: (2, 16, 192)
       tok_emb: (2, 16, 128)
         t_emb: (2, 128)
   conditioned: (2, 16, 128)
t_emb[:, None, :]: torch.Size([2, 1, 128])
✅ Time embedding + broadcast passed
